<a href="https://colab.research.google.com/github/asiahcrutchfield/ground-level-languages-ai/blob/main/gll_intro_to_ai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip install -q allosaurus torchcrepe librosa soundfile scipy numpy

import os, json, zipfile
import numpy as np
import torch
import torchcrepe
import librosa
import soundfile as sf
from scipy.signal import find_peaks
from allosaurus.app import read_recognizer

os.makedirs("audio/slices", exist_ok=True)
recognizer = read_recognizer()
device = "cuda" if torch.cuda.is_available() else "cpu"

IPA_VOWELS = set("aeiouyɨʉɯuɪʏʊeøɘɵɤoəɛœɜɞʌɔæɐaɶɑɒɚɝ")

def classify_phone(ipa_symbol):
    return "vowel" if any(ch in IPA_VOWELS for ch in ipa_symbol) else "consonant"

def synthesize_humming_track(f0, periodicity, sr=16000, hop_length=160):
    total_samples = int(len(f0)) * int(hop_length)
    freq_interp = np.interp(np.linspace(0, len(f0), total_samples), np.arange(len(f0)), f0)
    conf_interp = np.interp(np.linspace(0, len(periodicity), total_samples), np.arange(len(periodicity)), periodicity)

    amp = np.where(conf_interp > 0.35, 0.25, 0.0)
    phase = np.divide(np.multiply(2.0 * np.pi, np.cumsum(freq_interp)), sr)
    fundamental = np.multiply(0.8, np.sin(phase))
    harmonic = np.multiply(0.2, np.sin(np.multiply(2.0, phase)))
    return np.multiply(amp, fundamental + harmonic).astype(np.float32)

def process_word(word_id, mp3_path, visual_path):
    y, sr = librosa.load(mp3_path, sr=16000)
    temp_wav = f"/content/{word_id}_16k.wav"
    sf.write(temp_wav, y, sr)

    # 1. Allosaurus: Timestamped phoneme slicing (zero IPA sent to UI)
    raw_phones = recognizer.recognize(temp_wav, lang_id='eng', timestamp=True)
    phonemes_out = []
    for idx, line in enumerate(raw_phones.strip().split("\n")):
        if not line.strip():
            continue
        parts = line.split()
        start_s, dur_s, ipa = float(parts[0]), float(parts[1]), parts[2]
        s_idx = max(0, int((start_s - 0.015) * sr))
        e_idx = min(len(y), int((start_s + dur_s + 0.025) * sr))
        clip_path = f"audio/slices/{word_id}_p{idx+1}.wav"
        sf.write(clip_path, y[s_idx:e_idx], sr)
        phonemes_out.append({
            "id": f"{word_id}_p{idx+1}",
            "kind": classify_phone(ipa),
            "audio_url": clip_path
        })

    # 2. Syllable duration segmentation
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=160)[0]
    peaks, _ = find_peaks(rms, distance=12, prominence=np.max(rms) * 0.15)
    if len(peaks) == 0:
        peaks = [len(rms) // 2]
    boundaries = [0] + [int((peaks[i] + peaks[i+1]) / 2) for i in range(len(peaks) - 1)] + [len(rms)]
    syllables_out = []
    for idx in range(len(boundaries) - 1):
        s_s = (boundaries[idx] * 160) / sr
        e_s = (boundaries[idx+1] * 160) / sr
        dur = max(0.08, e_s - s_s)
        clip_path = f"audio/slices/{word_id}_s{idx+1}.wav"
        sf.write(clip_path, y[int(s_s * sr):int(e_s * sr)], sr)
        syllables_out.append({
            "id": f"{word_id}_s{idx+1}",
            "width_px": int(min(180, max(55, dur * 240))),
            "audio_url": clip_path
        })

    # 3. TorchCREPE: F0 pitch extraction & stripped humming synthesis
    audio_tensor = torch.tensor(y).unsqueeze(0)
    pitch, periodicity = torchcrepe.predict(
        audio_tensor,
        sr,
        hop_length=160,
        fmin=65.0,
        fmax=600.0,
        model="tiny",
        decoder=torchcrepe.decode.viterbi,
        return_periodicity=True,
        device=device
    )
    frequency = pitch.squeeze(0).cpu().numpy()
    conf = periodicity.squeeze(0).cpu().numpy()

    hum_out = f"audio/slices/{word_id}_hum.wav"
    sf.write(hum_out, synthesize_humming_track(frequency, conf, sr=sr, hop_length=160), sr)

    tones_out = []
    num_syl = len(syllables_out)
    for idx in range(num_syl):
        s_frame = int((idx / num_syl) * len(frequency))
        e_frame = int(((idx + 1) / num_syl) * len(frequency))
        seg_f0 = frequency[s_frame:e_frame]
        slope = float(seg_f0[-1] - seg_f0[0]) if len(seg_f0) > 1 else 0.0
        style = "solid" if slope > 12 else ("ring" if slope < -12 else "dot")
        tones_out.append({"id": f"{word_id}_t{idx+1}", "style": style, "audio_url": hum_out})

    return {
        "id": word_id,
        "visual_url": visual_path,
        "full_audio_url": f"audio/{word_id}.mp3",
        "humming_audio_url": hum_out,
        "phonemes": phonemes_out,
        "syllables": syllables_out,
        "tones": tones_out
    }

words = [
    ("cat", "/content/cat.mp3", "images/visuals/cat.jpg"),
    ("dog", "/content/dog.mp3", "images/visuals/dog.jpg"),
    ("walk", "/content/walk.mp3", "images/visuals/walk.gif"),
    ("jump", "/content/jump.mp3", "images/visuals/jump.gif"),
    ("tree", "/content/tree.mp3", "images/visuals/tree.jpg")
]

# Verify uploaded files before running
found_files = [p for _, p, _ in words if os.path.exists(p)]
print(f"Found {len(found_files)} audio files in /content/: {found_files}")

dataset = {"items": [process_word(w, p, v) for w, p, v in words if os.path.exists(p)]}

with open("lesson_data.json", "w") as f:
    json.dump(dataset, f, indent=2)

with zipfile.ZipFile("gll_update1_assets.zip", "w") as zf:
    zf.write("lesson_data.json")
    for root, _, files in os.walk("audio/slices"):
        for file in files:
            zf.write(os.path.join(root, file))

print(f"Done! Processed {len(dataset['items'])} words. Download gll_update1_assets.zip from the left sidebar.")

Found 5 audio files in /content/: ['/content/cat.mp3', '/content/dog.mp3', '/content/walk.mp3', '/content/jump.mp3', '/content/tree.mp3']
Done! Processed 5 words. Download gll_update1_assets.zip from the left sidebar.
